# 09_backtesting_and_risk_management.ipynb

This notebook runs the backtesting simulation using VectorBT and evaluates position sizing and risk management strategies.

### Objectives:
1. Load the trained meta-classifier and calibration models.
2. Generate out-of-sample trading signals on the test set.
3. Apply position sizing (Kelly Criterion or Volatility-adjusted).
4. Simulate trading using VectorBT (incorporating transaction costs and slippage).
5. Log trade history and calculate performance metrics (Sharpe, Sortino, Drawdown).

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Git Integration - Pull latest code from GitHub
import os
PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
GITHUB_REPO_URL = "https://github.com/umutergul74/TradeBot.git"

print("Pulling latest code from GitHub...")
os.chdir(PROJECT_ROOT)
# Initialize git and pull
!git init -b main
!git remote remove origin 2>/dev/null || true
!git remote add origin {GITHUB_REPO_URL}
!git fetch origin
!git reset --hard origin/main
print("Codebase updated to latest GitHub commit!")

In [ ]:
# Auto-install dependencies if any key package is missing
try:
    import ccxt
    import vectorbt
    import optuna
    import catboost
    import ta
    import shap
except ImportError:
    print("Some dependencies are missing. Installing from requirements.txt...")
    requirements_path = os.path.join(PROJECT_ROOT, "requirements.txt")
    !pip install -q -r {requirements_path}
    print("Installation complete!")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations & Test Data

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

# Load features and labels
features_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_5m_features.parquet")
df_features = load_parquet(features_path)
labels_path = os.path.join(PROJECT_ROOT, "data", "labels", f"{symbol}_5m_labels.parquet")
df_labels = load_parquet(labels_path)

df_model = df_features.loc[df_labels.index].copy()
df_model["label"] = df_labels["label"].astype(float)
df_model["meta_label"] = df_labels["meta_label"].astype(float)

# Split out-of-sample test set (last 20% of data)
test_idx = int(len(df_model) * 0.8)
df_test = df_model.iloc[test_idx:].copy()
print(f"Test dataset shape: {df_test.shape}")

## 3. Generate Trading Signals (Meta-Label Filtering)

In [ ]:
from models.model_registry import ModelRegistry

registry = ModelRegistry(PROJECT_ROOT)
model = registry.load_model(f"{symbol}_lgbm_model")
calibrator = registry.load_model(f"{symbol}_lgbm_calibrator")

# Dynamically extract features the model was trained on
features = list(model.model.feature_names_in_)
X_test = df_test[features]

raw_probs = model.predict_proba(X_test)
calibrated_probs = calibrator.calibrate(raw_probs)

# Class 1 probability represents the likelihood of the setup succeeding
# If probability is high, we trigger a signal (+1 for long setups, -1 for bearish setups)
signals = pd.Series(0, index=df_test.index)
threshold = 0.55

# To determine direction, we look at the confluence score (engineered feature)
is_bullish_setup = df_test["confluence_score"] > 0
is_bearish_setup = df_test["confluence_score"] < 0

signals = np.where(
    (calibrated_probs[:, 1] >= threshold) & is_bullish_setup,
    1,
    np.where(
        (calibrated_probs[:, 1] >= threshold) & is_bearish_setup,
        -1,
        0
    )
)
signals = pd.Series(signals, index=df_test.index)
print(f"Generated {signals.value_counts().get(1, 0)} long signals and {signals.value_counts().get(-1, 0)} short signals.")

## 4. Position Sizing

In [ ]:
from risk.position_sizing import KellySizer

# Apply Kelly criterion sizing based on probability
sizer = KellySizer(fractional_kelly=0.5)
sizes = []

for prob in calibrated_probs[:, 1]:
    # simple win rate and win/loss ratio (e.g. 2.0 RR)
    size = sizer.calculate_size(p_win=prob, win_loss_ratio=2.0)
    sizes.append(size)

df_test["position_size"] = sizes
print("Sample position sizes calculated:")
print(df_test["position_size"].describe())

## 5. Run Backtest Simulation

In [ ]:
import vectorbt as vbt

# Simulate the backtest using vectorbt
price = df_test["close"]
entries = signals == 1
exits = signals == -1

# Run simulation incorporating transaction fees (e.g. 0.04% taker fee on Binance Futures)
portfolio = vbt.Portfolio.from_signals(
    price,
    entries=entries,
    exits=exits,
    size=df_test["position_size"],
    fees=0.0004,
    slippage=0.0005,
    init_cash=10000.0
)

print(portfolio.stats())

## 6. Save Backtest Results

In [ ]:
from utils.io_utils import save_parquet

df_trades = portfolio.trades.records_readable
if not df_trades.empty:
    trade_journal_path = os.path.join(PROJECT_ROOT, "reports", "backtests", f"{symbol}_trade_journal.parquet")
    save_parquet(df_trades, trade_journal_path)
    print(f"Trade journal saved to {trade_journal_path}")
else:
    print("No trades were executed during backtest.")

## Summary of Backtest Results

We completed:
1. VectorBT backtest simulation on the out-of-sample test set.
2. Saved the detailed trade journal to `reports/backtests/`.

**Next Step**: Run [10_explainability_and_trade_journal.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/10_explainability_and_trade_journal.ipynb) to explain the model decisions and build the trade logger.

In [ ]:
# Auto-disconnect Colab runtime to save compute units
AUTO_DISCONNECT = False  # Set to True to enable automatic shutdown
if AUTO_DISCONNECT:
    print("Disconnecting Colab runtime...")
    from google.colab import runtime
    runtime.unassign()